# Baseline A vs Baseline B — Comparison Report
## Week 3 | xBD Track | Disaster Damage Assessment
### ResNet50 CNN (post-only) vs Change-Detection (pre+post)


In [ ]:
# Cell 2: Setup
import os
import json
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print('Not in Colab, skipping drive mount.')

import sys
sys.path.append('/content/drive/MyDrive/Project/disaSense_project')

# Fallback for local run
if not os.path.exists('/content/drive/MyDrive/Project/disaSense_project/configs/config.yaml'):
    sys.path.append('..')
    config_path = '../configs/config.yaml'
else:
    config_path = '/content/drive/MyDrive/Project/disaSense_project/configs/config.yaml'

# Load config
with open(config_path) as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

class_names = ["no-damage", "minor-damage", "major-damage", "destroyed"]
DAMAGE_COLORS = {
    "no-damage": "green",
    "minor-damage": "gold", 
    "major-damage": "orange",
    "destroyed": "red"
}

results_dir = os.path.join(config["paths"]["results_dir"], "baselines")
os.makedirs(results_dir, exist_ok=True)


In [ ]:
# Cell 3: Load test dataset
try:
    from src.preprocessing.patch_cropper import XBDPatchDataset
    print("Loading test dataset...")
    test_dataset = XBDPatchDataset(config["paths"]["xbd_root"], split="test", seed=config["project"]["seed"])
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    print("\nTest set summary:")
    summary = test_dataset.get_split_summary()
    print(f"Total images: {summary['n_image_pairs']}")
    print(f"Total buildings: {summary['n_buildings']}")
    print("\nSamples per class:")
    for k, v in summary['class_distribution'].items():
        print(f"  {k}: {v}")
    print("\nSamples per disaster type:")
    for k, v in summary['disaster_type_distribution'].items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"Error loading dataset: {e}")


In [ ]:
# Cell 4: Load and evaluate Baseline A
from src.models.baseline_a_cnn import ResNet50DamageClassifier, evaluate_baseline_a, load_checkpoint

ckpt_a_path = os.path.join(config["paths"]["checkpoint_dir"], "baseline_a_best.pt")
results_a_path = os.path.join(results_dir, "baseline_a_test_results.json")

results_a = None
if not os.path.exists(ckpt_a_path):
    print(f"Warning: Baseline A checkpoint not found at {ckpt_a_path}. Skipping evaluation.")
else:
    try:
        print(f"Loading Baseline A from {ckpt_a_path}...")
        model_a = ResNet50DamageClassifier().to(device)
        model_a, _, _, _ = load_checkpoint(ckpt_a_path, model_a)
        
        print("Evaluating Baseline A on test set...")
        results_a = evaluate_baseline_a(model_a, test_loader, device, class_names)
        
        # Save results to json (exclude numpy arrays)
        json_results_a = {
            "accuracy": results_a["accuracy"],
            "macro_f1": results_a["macro_f1"],
            "variance_score": results_a["variance_score"],
            "per_class_f1": results_a["per_class_f1"]
        }
        with open(results_a_path, 'w') as f:
            json.dump(json_results_a, f, indent=4)
            
        print("\nModel: Baseline A (ResNet50 CNN — post-disaster only)")
        print(f"Accuracy:       {results_a['accuracy']:.4f}")
        print(f"Macro-F1:       {results_a['macro_f1']:.4f}")
        print(f"Variance Score: {results_a['variance_score']:.4f}  (std of per-class F1 — lower is better)")
        print("Per-class F1:")
        for cls in class_names:
            print(f"  {cls}: {results_a['per_class_f1'][cls]:.4f}")
    except Exception as e:
        print(f"Error evaluating Baseline A: {e}")


In [ ]:
# Cell 5: Load and evaluate Baseline B
from src.models.baseline_b_change import ChangeDetectionClassifier, evaluate_baseline_b

ckpt_b_path = os.path.join(config["paths"]["checkpoint_dir"], "baseline_b_best.pt")
results_b_path = os.path.join(results_dir, "baseline_b_test_results.json")

results_b = None
if not os.path.exists(ckpt_b_path):
    print(f"Warning: Baseline B checkpoint not found at {ckpt_b_path}. Skipping evaluation.")
else:
    try:
        print(f"Loading Baseline B from {ckpt_b_path}...")
        model_b = ChangeDetectionClassifier(pretrained=False).to(device)
        model_b, _, _, _ = load_checkpoint(ckpt_b_path, model_b)
        
        print("Evaluating Baseline B on test set...")
        results_b = evaluate_baseline_b(model_b, test_loader, device, class_names)
        
        # Save results to json
        json_results_b = {
            "accuracy": results_b["accuracy"],
            "macro_f1": results_b["macro_f1"],
            "variance_score": results_b["variance_score"],
            "per_class_f1": results_b["per_class_f1"]
        }
        with open(results_b_path, 'w') as f:
            json.dump(json_results_b, f, indent=4)
            
        print("\nModel: Baseline B (Change-Detection — pre+post)")
        print(f"Accuracy:       {results_b['accuracy']:.4f}")
        print(f"Macro-F1:       {results_b['macro_f1']:.4f}")
        print(f"Variance Score: {results_b['variance_score']:.4f}  (std of per-class F1 — lower is better)")
        print("Per-class F1:")
        for cls in class_names:
            print(f"  {cls}: {results_b['per_class_f1'][cls]:.4f}")
    except Exception as e:
        print(f"Error evaluating Baseline B: {e}")


In [ ]:
# Cell 6: Side-by-side metric comparison table
if results_a and results_b:
    table_data = []
    
    def add_row(metric_name, val_a, val_b):
        delta = val_b - val_a
        winner = "B" if val_b > val_a else "A"
        if metric_name == "Variance Score":
            winner = "B" if val_b < val_a else "A"
        table_data.append([metric_name, val_a, val_b, delta, winner])
        
    add_row("Accuracy", results_a["accuracy"], results_b["accuracy"])
    add_row("Macro-F1", results_a["macro_f1"], results_b["macro_f1"])
    for cls in class_names:
        add_row(f"F1 ({cls})", results_a["per_class_f1"][cls], results_b["per_class_f1"][cls])
    add_row("Variance Score", results_a["variance_score"], results_b["variance_score"])
    
    df = pd.DataFrame(table_data, columns=["Metric", "Baseline A (CNN)", "Baseline B (Change-Det)", "Delta", "Winner"])
    
    print("| Metric | Baseline A (CNN) | Baseline B (Change-Det) | Delta |")
    print("|---|---|---|---|")
    for row in table_data:
        winner_mark = "*" if row[4] == "B" else ""
        delta_str = f"+{row[3]:.4f}" if row[3] > 0 else f"{row[3]:.4f}"
        print(f"| {row[0]:<19} | {row[1]:.4f} | {row[2]:.4f} | {delta_str} |")
        
    macro_f1_a = results_a["macro_f1"]
    macro_f1_b = results_b["macro_f1"]
    winner = "Baseline B (Change-Detection)" if macro_f1_b > macro_f1_a else "Baseline A (CNN)"
    print(f"\nWinner by Macro-F1: {winner}")
    
    csv_path = os.path.join(results_dir, "ab_comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved comparison table to {csv_path}")
else:
    print("Cannot create comparison table: missing results for A or B.")


In [ ]:
# Cell 7: Grouped bar chart comparison
if results_a and results_b:
    x = np.arange(len(class_names))
    width = 0.35
    
    a_f1s = [results_a["per_class_f1"][cls] for cls in class_names]
    b_f1s = [results_b["per_class_f1"][cls] for cls in class_names]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, a_f1s, width, label='Baseline A (CNN)', color='tab:blue')
    rects2 = ax.bar(x + width/2, b_f1s, width, label='Baseline B (Change-Detection)', color='tab:orange')
    
    ax.set_ylabel('F1 Score (0 to 1)')
    ax.set_title('Per-Class F1 Comparison: Baseline A vs Baseline B')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.legend()
    
    # Horizontal lines for Macro-F1
    ax.axhline(results_a["macro_f1"], color='tab:blue', linestyle='--', alpha=0.7)
    ax.axhline(results_b["macro_f1"], color='tab:orange', linestyle='--', alpha=0.7)
    
    ax.set_ylim([0, 1.05])
    
    plt.tight_layout()
    chart_path = os.path.join(results_dir, "ab_f1_comparison.png")
    plt.savefig(chart_path, dpi=150)
    plt.show()
else:
    print("Cannot create bar chart: missing results for A or B.")


In [ ]:
# Cell 8: Side-by-side confusion matrices
if results_a and results_b:
    cm_a = np.array(results_a["confusion_matrix"])
    cm_b = np.array(results_b["confusion_matrix"])
    
    # Normalize by row
    cm_a_norm = cm_a.astype('float') / cm_a.sum(axis=1)[:, np.newaxis]
    cm_a_norm = np.nan_to_num(cm_a_norm)
    cm_b_norm = cm_b.astype('float') / cm_b.sum(axis=1)[:, np.newaxis]
    cm_b_norm = np.nan_to_num(cm_b_norm)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.heatmap(cm_a_norm * 100, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Baseline A — Confusion Matrix (Test Set)')
    axes[0].set_xlabel('Predicted class')
    axes[0].set_ylabel('True class')
    
    sns.heatmap(cm_b_norm * 100, annot=True, fmt='.1f', cmap='Oranges',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('Baseline B — Confusion Matrix (Test Set)')
    axes[1].set_xlabel('Predicted class')
    axes[1].set_ylabel('True class')
    
    plt.tight_layout()
    cm_path = os.path.join(results_dir, "ab_confusion_matrices.png")
    plt.savefig(cm_path, dpi=150)
    plt.show()
else:
    print("Cannot create confusion matrices: missing results for A or B.")


In [ ]:
# Cell 9: Variance score analysis
if results_a and results_b:
    var_a = results_a["variance_score"]
    var_b = results_b["variance_score"]
    
    better_model = "Baseline A" if var_a < var_b else "Baseline B"
    diff = abs(var_a - var_b)
    
    print("=== VARIANCE SCORE ANALYSIS ===")
    print(f"Baseline A variance score: {var_a:.4f}")
    print(f"Baseline B variance score: {var_b:.4f}")
    print("\nLower variance score = more balanced performance across damage classes.")
    print("A model with high overall accuracy but high variance score is succeeding")
    print("on easy classes (no-damage, destroyed) while failing on hard ones")
    print("(minor-damage, major-damage).")
    print(f"\nObservation: {better_model} has a lower variance score by {diff:.4f}.")
    print("\nThis is the key metric our ensemble aims to improve further.")
    print(f"Target: Reduce variance score below {min(var_a, var_b):.4f}")
    print("        via weighted ensemble in Phase 4.")
else:
    print("Cannot perform variance analysis: missing results for A or B.")


In [ ]:
# Cell 10: Error analysis: where do both models fail?
if results_a and results_b:
    preds_a = np.array(results_a["all_predictions"])
    preds_b = np.array(results_b["all_predictions"])
    labels = np.array(results_a["all_labels"])
    
    both_wrong = (preds_a != labels) & (preds_b != labels)
    wrong_indices = np.where(both_wrong)[0]
    
    print(f"Total samples where both are wrong: {len(wrong_indices)}")
    
    wrong_labels = labels[wrong_indices]
    print("\nBreakdown by true damage class:")
    for i, cls in enumerate(class_names):
        count = np.sum(wrong_labels == i)
        print(f"  {cls}: {count}")
        
    print("\nBreakdown by disaster type:")
    disaster_counts = {}
    for idx in wrong_indices:
        disaster = test_dataset.building_index[idx]["disaster_type"]
        disaster_counts[disaster] = disaster_counts.get(disaster, 0) + 1
        
    for d, c in sorted(disaster_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {d}: {c}")
        
    print("\nDisplaying 3 examples where both models failed:")
    examples_to_show = min(3, len(wrong_indices))
    
    for i in range(examples_to_show):
        idx = wrong_indices[i]
        sample_meta = test_dataset.building_index[idx]
        true_label_name = class_names[labels[idx]]
        pred_a_name = class_names[preds_a[idx]]
        pred_b_name = class_names[preds_b[idx]]
        
        try:
            pre_img = Image.open(sample_meta["pre_image_path"])
            post_img = Image.open(sample_meta["post_image_path"])
            
            x, y, w, h = sample_meta["bbox"]
            alpha = test_dataset.alpha
            exp_w, exp_h = w + alpha * w, h + alpha * h
            cx, cy = x + w/2, y + h/2
            crop_box = (
                max(0, int(cx - exp_w/2)),
                max(0, int(cy - exp_h/2)),
                min(pre_img.width, int(cx + exp_w/2)),
                min(pre_img.height, int(cy + exp_h/2))
            )
            pre_crop = pre_img.crop(crop_box)
            post_crop = post_img.crop(crop_box)
            
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(pre_crop)
            axes[0].set_title("Pre-Disaster")
            axes[0].axis('off')
            
            axes[1].imshow(post_crop)
            axes[1].set_title("Post-Disaster")
            axes[1].axis('off')
            
            plt.suptitle(f"Both models wrong: True=[{true_label_name}], A=[{pred_a_name}], B=[{pred_b_name}]", y=1.05)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Error displaying image for example {i}: {e}")
else:
    print("Cannot perform error analysis: missing results for A or B.")


In [ ]:
# Cell 11: Observation and insight for faculty meeting
if results_a and results_b:
    better_overall = "Baseline B" if results_b["macro_f1"] > results_a["macro_f1"] else "Baseline A"
    diff_overall = abs(results_b["macro_f1"] - results_a["macro_f1"])
    
    avg_f1s = [(results_a["per_class_f1"][cls] + results_b["per_class_f1"][cls])/2 for cls in class_names]
    hardest_idx = np.argmin(avg_f1s)
    hardest_class = class_names[hardest_idx]
    
    b_beats_a_on_minor = results_b["per_class_f1"]["minor-damage"] > results_a["per_class_f1"]["minor-damage"]
    minor_diff = results_b["per_class_f1"]["minor-damage"] - results_a["per_class_f1"]["minor-damage"]
    
    more_balanced = "Baseline B" if results_b["variance_score"] < results_a["variance_score"] else "Baseline A"
    var_diff = abs(results_b["variance_score"] - results_a["variance_score"])
    
    preds_a = np.array(results_a["all_predictions"])
    preds_b = np.array(results_b["all_predictions"])
    labels = np.array(results_a["all_labels"])
    errors_a = (preds_a != labels)
    errors_b = (preds_b != labels)
    exclusive_errors = np.sum(errors_a ^ errors_b)
    total_errors = np.sum(errors_a | errors_b)
    exclusive_pct = (exclusive_errors / total_errors) * 100 if total_errors > 0 else 0
    
    print("=== KEY OBSERVATIONS FOR FACULTY MEETING ===\n")
    print(f"1. OVERALL PERFORMANCE:")
    print(f"   {better_overall} performs better overall by {diff_overall:.4f} Macro-F1.")
    
    print(f"\n2. CLASS-LEVEL INSIGHT:")
    print(f"   {hardest_class} is the hardest damage class for both models to classify.")
    
    print(f"\n3. CHANGE-DETECTION BENEFIT:")
    if b_beats_a_on_minor:
        print(f"   Yes, Baseline B outperforms A on 'minor-damage' by {minor_diff:.4f}, proving temporal context helps.")
    else:
        print(f"   No, Baseline B does not outperform A on 'minor-damage' (difference: {minor_diff:.4f}).")
        
    print(f"\n4. VARIANCE:")
    print(f"   {more_balanced} is more balanced across all classes by {var_diff:.4f} variance score.")
    
    print(f"\n5. ENSEMBLE HYPOTHESIS:")
    print(f"   \"Since Baseline A and Baseline B fail on different samples")
    print(f"   (evidenced by {exclusive_pct:.1f}% of errors being exclusive to one model),")
    print(f"   combining them via weighted voting is expected to reduce")
    print(f"   per-class variance further. This will be tested in Week 4.\"")
else:
    print("Cannot generate observations: missing results for A or B.")


In [ ]:
# Cell 12: Save complete report
if results_a and results_b:
    report_data = {
        "baseline_a_results": results_a_path,
        "baseline_b_results": results_b_path,
        "comparison_table": os.path.join(results_dir, "ab_comparison_table.csv"),
        "f1_comparison_chart": os.path.join(results_dir, "ab_f1_comparison.png"),
        "confusion_matrices": os.path.join(results_dir, "ab_confusion_matrices.png"),
        "key_observation": f"{better_overall} wins overall, but {exclusive_pct:.1f}% of errors are mutually exclusive, supporting ensemble hypothesis."
    }
    
    report_path = os.path.join(results_dir, "week3_faculty_report.json")
    with open(report_path, "w") as f:
        json.dump(report_data, f, indent=4)
        
    print(f"Week 3 report complete. All outputs saved to {report_path}.")
else:
    print("Report incomplete due to missing baseline results.")
